# Threat Intelligence Module

Advanced threat intelligence processing with IOC extraction, APT tracking, and risk scoring.

## Contents

1. **IOC Extraction** - Extract indicators of compromise
2. **APT Tracking** - Advanced persistent threat profiles
3. **Risk Scoring** - Threat severity calculation
4. **Enrichment** - External threat intel API integration


## 1. IOC Extraction Module


In [ ]:
import re
import json
import hashlib
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Set
from enum import Enum
from datetime import datetime
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class IOCType(Enum):
    """Types of Indicators of Compromise."""
    IPV4 = "IPv4"
    IPV6 = "IPv6"
    DOMAIN = "Domain"
    URL = "URL"
    EMAIL = "Email"
    HASH_MD5 = "MD5"
    HASH_SHA1 = "SHA1"
    HASH_SHA256 = "SHA256"
    HASH_SHA512 = "SHA512"
    USER_AGENT = "User-Agent"
    ASN = "ASN"
    BTC_ADDRESS = "Bitcoin"
    REGISTRY_KEY = "Registry"
    FILE_PATH = "FilePath"
    MUTEX = "Mutex"


@dataclass
class IOC:
    """Indicator of Compromise."""
    ioc_type: IOCType
    value: str
    context: str = ""
    confidence: float = 1.0
    source: str = ""
    first_seen: Optional[datetime] = None
    last_seen: Optional[datetime] = None
    tags: Set[str] = field(default_factory=set)


class IOCExtractor:
    """Extract IOCs from text/blob data."""
    
    PATTERNS = {
        IOCType.IPV4: [
            r"\b(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\b"
        ],
        IOCType.IPV6: [
            r"\b(?:[0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}\b",
            r"\b(?:[0-9a-fA-F]{1,4}:){1,7}:\b",
            r"::(?:[fF]{4}:)?(?:(?:25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])\.){3}(?:25[0-5]|2[0-4][0-9]|1[0-9][0-9]|[1-9]?[0-9])",
        ],
        IOCType.DOMAIN: [
            r"\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,}\b",
            r"\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,6}\b",
        ],
        IOCType.URL: [
            r"https?:\/\/(?:www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b(?:[-a-zA-Z0-9()@:%_\+.~#?&\/=]*)",
            r"ftp:\/\/[-a-zA-Z0-9@:%._\+~#=]{1,256}\.[a-zA-Z0-9()]{1,6}\b(?:[-a-zA-Z0-9()@:%_\+.~#?&\/=]*)",
        ],
        IOCType.EMAIL: [
            r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"
        ],
        IOCType.HASH_MD5: [
            r"\b[a-fA-F0-9]{32}\b"
        ],
        IOCType.HASH_SHA1: [
            r"\b[a-fA-F0-9]{40}\b"
        ],
        IOCType.HASH_SHA256: [
            r"\b[a-fA-F0-9]{64}\b"
        ],
        IOCType.HASH_SHA512: [
            r"\b[a-fA-F0-9]{128}\b"
        ],
        IOCType.ASN: [
            r"\bAS\d{1,10}\b",
            r"\bASn\s*\d{1,10}\b"
        ],
        IOCType.BTC_ADDRESS: [
            r"\b[13][a-km-zA-HJ-NP-Z1-9]{25,34}\b",
            r"\b[mn][1-9A-HJ-NP-Za-km-z1-9]{25,34}\b"
        ],
        IOCType.REGISTRY_KEY: [
            r"(?:HKLM|HKCU|HKCR|HKU|HKCC)\\\\(?:[A-Za-z0-9_]+\\\\)*[A-Za-z0-9_]+",
            r"HKEY_LOCAL_MACHINE\\\\[A-Za-z0-9_]+",
        ],
        IOCType.FILE_PATH: [
            r"[A-Za-z]:\\\\[A-Za-z0-9_\\\\.]+",
            r"\/home\/[A-Za-z0-9_]+\/[A-Za-z0-9_\/\.]+",
            r"\/usr\/bin\/[A-Za-z0-9_]+",
            r"C:\\Users\\[A-Za-z0-9_]+\\AppData\\[A-Za-z0-9_\\]+",
        ],
        IOCType.MUTEX: [
            r"Global\\\\[A-Za-z0-9_]+",
            r"Local\\\\[A-Za-z0-9_]+",
        ],
    }
    
    # Exclude common false positives
    FALSE_POSITIVES = {
        "example.com", "example.org", "test.com", "localhost",
        "127.0.0.1", "0.0.0.0", "255.255.255.255",
    }
    
    def __init__(self):
        self.compiled: Dict[IOCType, List[re.Pattern]] = {}
        self._compile_patterns()
    
    def _compile_patterns(self):
        for ioc_type, patterns in self.PATTERNS.items():
            self.compiled[ioc_type] = [
                re.compile(p, re.IGNORECASE) for p in patterns
            ]
    
    def extract(self, text: str) -> List[IOC]:
        """Extract IOCs from text."""
        iocs = []
        
        for ioc_type, patterns in self.compiled.items():
            for pattern in patterns:
                for match in pattern.finditer(text):
                    value = match.group()
                    
                    # Skip false positives
                    if value.lower() in self.FALSE_POSITIVES:
                        continue
                    
                    # Calculate confidence based on context
                    confidence = self._calculate_confidence(text, match, ioc_type)
                    
                    # Skip low confidence
                    if confidence < 0.5:
                        continue
                    
                    iocs.append(IOC(
                        ioc_type=ioc_type,
                        value=value,
                        context=text[max(0, match.start()-50):match.end()+50],
                        confidence=confidence
                    ))
        
        # Remove duplicates
        seen = set()
        unique = []
        for ioc in iocs:
            key = (ioc.ioc_type, ioc.value.lower())
            if key not in seen:
                seen.add(key)
                unique.append(ioc)
        
        return unique
    
    def _calculate_confidence(self, text: str, match: re.Match, ioc_type: IOCType) -> float:
        """Calculate extraction confidence."""
        base = 0.7
        
        # Check for surrounding context that indicates IOCs
        context = text[max(0, match.start()-100):match.end()+100]
        context_lower = context.lower()
        
        # High confidence indicators
        high_confidence = [
            "malware", "c2", "command and control", "payload",
            "indicator", "threat", "apt", "campaign"
        ]
        if any(ind in context_lower for ind in high_confidence):
            base = 0.95
        
        # Medium confidence indicators
        medium_confidence = [
            "ip", "domain", "server", "host", "found", "detected"
        ]
        if any(ind in context_lower for ind in medium_confidence):
            base = max(base, 0.8)
        
        # Adjust based on hash type certainty
        if ioc_type in [IOCType.HASH_MD5, IOCType.HASH_SHA1, IOCType.HASH_SHA256, IOCType.HASH_SHA512]:
            if len(match.group()) == 32 and ioc_type == IOCType.HASH_MD5:
                base = max(base, 0.9)
            elif len(match.group()) == 40 and ioc_type == IOCType.HASH_SHA1:
                base = max(base, 0.9)
            elif len(match.group()) == 64 and ioc_type == IOCType.HASH_SHA256:
                base = max(base, 0.9)
        
        return min(1.0, base)


class ThreatIntelligenceEnricher:
    """Enrich IOCs with external threat intelligence."""
    
    def __init__(self, api_keys: Dict[str, str] = None):
        """Initialize enricher."""
        self.api_keys = api_keys or {}
        self._cache = {}
    
    def enrich(self, ioc: IOC) -> Dict:
        """Enrich IOC with threat intel."""
        enrichment = {
            "ioc": ioc.value,
            "type": ioc.ioc_type.value,
            "reputation": None,
            "tags": list(ioc.tags),
            "enrichment_sources": []
        }
        
        if ioc.ioc_type in [IOCType.IPV4, IOCType.IPV6]:
            enrichment.update(self._enrich_ip(ioc.value))
        elif ioc.ioc_type == IOCType.DOMAIN:
            enrichment.update(self._enrich_domain(ioc.value))
        elif ioc.ioc_type in [IOCType.HASH_MD5, IOCType.HASH_SHA1, IOCType.HASH_SHA256]:
            enrichment.update(self._enrich_hash(ioc.value))
        
        return enrichment
    
    def _enrich_ip(self, ip: str) -> Dict:
        """Enrich IP address with intel."""
        return {
            "geo": {
                "country": "Unknown",
                "city": "Unknown",
                "asn": "Unknown"
            },
            "reputation": "unknown",
            "enrichment_sources": ["local"]
        }
    
    def _enrich_domain(self, domain: str) -> Dict:
        """Enrich domain with intel."""
        return {
            "registration": {
                "registrar": "Unknown",
                "created": "Unknown"
            },
            "reputation": "unknown",
            "enrichment_sources": ["local"]
        }
    
    def _enrich_hash(self, hash_value: str) -> Dict:
        """Enrich file hash with intel."""
        return {
            "first_seen": "Unknown",
            "last_seen": "Unknown",
            "detection_rate": "Unknown",
            "reputation": "unknown",
            "enrichment_sources": ["local"]
        }


@dataclass
class APTProfile:
    """Advanced Persistent Threat profile."""
    name: str
    aliases: List[str]
    country: str
    first_seen: str
    targets: List[str]
    attack_types: List[str]
    malware_families: List[str]
    infrastructure: List[str]
    severity: str = "HIGH"


class APTTracker:
    """Track and identify APT groups."""
    
    KNOWN_APTS = {
        "APT28": APTProfile(
            name="APT28 (Fancy Bear)",
            aliases=["Fancy Bear", "STRONTIUM", "Sofacy"],
            country="Russia",
            first_seen="2004",
            targets=["Government", "Military", "Media", "Election"],
            attack_types=["Spearphishing", "Zero-day exploits", "Password spraying"],
            malware_families=["Sofacy", "X-Agent", "Gamefish"],
            infrastructure=[]
        ),
        "APT29": APTProfile(
            name="APT29 (Cozy Bear)",
            aliases=["Cozy Bear", "NOBELIUM", "YTTRIUM"],
            country="Russia",
            first_seen="2008",
            targets=["Government", "Think tanks", "Diplomacy"],
            attack_types=["Spearphishing", "Supply chain"],
            malware_families=["Sunburst", "Teardrop", "Rainday"],
            infrastructure=[]
        ),
        "APT41": APTProfile(
            name="APT41",
            aliases=["BARIUM", "Winnti", "Wicked Panda"],
            country="China",
            first_seen="2012",
            targets=["Gaming", "Healthcare", "Technology", "Government"],
            attack_types=["Supply chain", "Zero-day", "Cryptojacking"],
            malware_families=["Winnti", "PlugX", "ShadowPad"],
            infrastructure=[]
        ),
        "Lazarus": APTProfile(
            name="Lazarus Group",
            aliases=["Hidden Cobra", "Zinc"],
            country="North Korea",
            first_seen="2009",
            targets=["Finance", "Cryptocurrency", "Defense"],
            attack_types=["SWIFT attacks", "Spearphishing"],
            malware_families=["Destover", "WannaCry", "Joanap"],
            infrastructure=[]
        ),
    }
    
    def identify(self, iocs: List[IOC]) -> List[Tuple[IOC, List[APTProfile]]]:
        """Identify APTs based on IOCs."""
        matches = []
        
        for ioc in iocs:
            apt_matches = []
            
            for apt_name, apt in self.KNOWN_APTS.items():
                # Check infrastructure
                if ioc.ioc_type in [IOCType.IPV4, IOCType.IPV6, IOCType.DOMAIN]:
                    if ioc.value in apt.infrastructure:
                        apt_matches.append(apt)
                
                # Check malware families
                if ioc.ioc_type == IOCType.HASH_MD5 or ioc.ioc_type == IOCType.HASH_SHA256:
                    if ioc.value in apt.malware_families:
                        apt_matches.append(apt)
            
            if apt_matches:
                matches.append((ioc, apt_matches))
        
        return matches


@dataclass
class ThreatScore:
    """Threat severity score."""
    overall: float
    severity: str
    confidence: float
    breakdown: Dict


class ThreatScorer:
    """Calculate threat severity scores."""
    
    SEVERITY_WEIGHTS = {
        IOCType.HASH_MD5: 0.3,
        IOCType.HASH_SHA1: 0.3,
        IOCType.HASH_SHA256: 0.4,
        IOCType.HASH_SHA512: 0.4,
        IOCType.IPV4: 0.5,
        IOCType.IPV6: 0.5,
        IOCType.DOMAIN: 0.6,
        IOCType.URL: 0.8,
        IOCType.C2_SERVER: 0.95,
        IOCType.PHISHING_URL: 0.9,
    }
    
    def calculate(
        self,
        iocs: List[IOC],
        apt_matches: List[Tuple[IOC, List[APTProfile]]],
        enrichment: Dict[str, Dict]
    ) -> ThreatScore:
        """Calculate overall threat score."""
        breakdown = {
            "ioc_count": len(iocs),
            "apt_matches": len(apt_matches),
            "high_confidence": sum(1 for i in iocs if i.confidence > 0.8),
            "severity_distribution": {}
        }
        
        total_score = 0.0
        max_possible = 0.0
        
        for ioc in iocs:
            weight = self.SEVERITY_WEIGHTS.get(ioc.ioc_type, 0.5)
            max_possible += weight
            
            score = weight * ioc.confidence
            
            # Boost for APT matches
            for ioc_match, apts in apt_matches:
                if ioc_match.value == ioc.value:
                    score *= 1.5
            
            # Boost for malicious enrichment
            if ioc.value in enrichment:
                rep = enrichment[ioc.value].get("reputation", "")
                if rep in ["malicious", "suspicious"]:
                    score *= 1.3
            
            total_score += min(score, weight)
        
        if max_possible == 0:
            return ThreatScore(0, "LOW", 0.0, breakdown)
        
        normalized = min(total_score / max_possible, 1.0) * 100
        
        if normalized < 25:
            severity = "LOW"
        elif normalized < 50:
            severity = "MEDIUM"
        elif normalized < 75:
            severity = "HIGH"
        else:
            severity = "CRITICAL"
        
        return ThreatScore(
            overall=normalized,
            severity=severity,
            confidence=sum(i.confidence for i in iocs) / len(iocs) if iocs else 0,
            breakdown=breakdown
        )


## 2. Threat Intelligence Pipeline Demo


In [ ]:
def demonstrate_threat_intelligence():
    """Demonstrate the threat intelligence module."""
    
    # Sample threat report text
    threat_report = """
    THREAT INTELLIGENCE REPORT - APT Campaign Analysis
    
    The following indicators of compromise were identified in a recent attack:
    
    C2 Infrastructure:
    - 185.220.101.45 (Tor exit node)
    - 91.121.87.18
    - command-control.bad-domain.com
    - update.malware-site.net
    
    Malicious Files:
    - Payload: 5d41402abc4b2a76b9719d911017c592 (MD5)
    - Backdoor: a94a8fe5ccb19ba61c4c0873d391e987982fbd84 (SHA1)
    - Trojan: ef92b778bafe771e89245b89ecbc08a44a4e166c0666023 (SHA256)
    
    Additional IOCs:
    - Attacker email: attacker@evil-domain.org
    - C2 URL: https://malware-site.net/api/connect
    - AS Number: AS16276
    
    The campaign targets financial institutions in Europe.
    """
    
    print("=" * 70)
    print("THREAT INTELLIGENCE MODULE DEMO")
    print("=" * 70)
    
    # Step 1: Extract IOCs
    print("\n[1] Extracting IOCs...")
    extractor = IOCExtractor()
    iocs = extractor.extract(threat_report)
    
    print(f"    Found {len(iocs)} IOCs:")
    for ioc in iocs:
        print(f"      - [{ioc.ioc_type.value}] {ioc.value[:50]} (conf: {ioc.confidence:.2f})")
    
    # Step 2: Enrich IOCs
    print("\n[2] Enriching IOCs...")
    enricher = ThreatIntelligenceEnricher()
    enrichment_data = {}
    
    for ioc in iocs:
        data = enricher.enrich(ioc)
        enrichment_data[ioc.value] = data
        print(f"      - {ioc.value[:40]}: {data.get('reputation', 'unknown')}")
    
    # Step 3: Identify APTs
    print("\n[3] Identifying APTs...")
    apt_tracker = APTTracker()
    apt_matches = apt_tracker.identify(iocs)
    
    if apt_matches:
        print(f"    Found {len(apt_matches)} potential APT links:")
        for ioc, apts in apt_matches:
            print(f"      - {ioc.value[:40]}: {', '.join(a.name for a in apts)}")
    else:
        print("    No known APT matches (this may be a new threat)")
    
    # Step 4: Calculate threat score
    print("\n[4] Calculating Threat Score...")
    scorer = ThreatScorer()
    score = scorer.calculate(iocs, apt_matches, enrichment_data)
    
    print(f"\n    OVERALL THREAT SCORE: {score.overall:.1f}/100")
    print(f"    SEVERITY LEVEL: {score.severity}")
    print(f"    CONFIDENCE: {score.confidence:.2f}")
    print(f"\n    Breakdown:")
    print(f"      - Total IOCs: {score.breakdown['ioc_count']}")
    print(f"      - APT Links: {score.breakdown['apt_matches']}")
    print(f"      - High Confidence: {score.breakdown['high_confidence']}")
    
    return {
        "iocs": iocs,
        "enrichment": enrichment_data,
        "apt_matches": apt_matches,
        "threat_score": score
    }


if __name__ == "__main__":
    demonstrate_threat_intelligence()

## 3. Integration with OSINT Vortex

The threat intelligence module integrates with the OSINT Vortex pipeline:

```python
from osint_vortex import VortexPipeline
from threat_intel import IOCExtractor, ThreatScorer

pipeline = VortexPipeline()

# Add threat intel processing
pipeline.add_processor(IOCExtractor())
pipeline.add_processor(ThreatScorer())

# Process threat reports
results = pipeline.run("threat_reports/*.txt")
```


## 4. Summary

This notebook demonstrates:

- **IOC Extraction**: Regex-based detection of 15+ IOC types
- **Threat Enrichment**: Integration with external threat intel
- **APT Tracking**: Identification of known APT groups
- **Risk Scoring**: Quantified threat severity (0-100)

The module provides enterprise-grade threat intelligence processing capabilities.
